# 02 — Build and Run an Agent

This notebook walks you through interacting with a deployed agent or creating one from scratch.
Run cells top-to-bottom. Change `AGENT_NAME` below to work with any registered agent.

## Configuration

In [ ]:
# Change this to work with a different agent
AGENT_NAME = "code-helper"

## Install Dependencies

In [ ]:
!uv sync

## Connect to Foundry

In [ ]:
from dotenv import load_dotenv
import os
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

# Load endpoint from .env
load_dotenv()
ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]

project_client = AIProjectClient(
    endpoint=ENDPOINT,
    credential=DefaultAzureCredential(),
)
print("✓ Connected to Azure AI Foundry via AIProjectClient")

## Step 1: Get or Deploy the Agent

Uses the existing deployed agent if found, otherwise deploys it automatically via the registry.

### Integration Toggles (Optional)

Each agent controls its own tools via its `config.py`. Use the toggles below to override at runtime for quick testing. To persist changes, update the agent's `config.py` directly.

In [ ]:
# Toggle integrations — set to True/False as needed
CODE_INTERPRETER_ENABLED = False
WEB_SEARCH_ENABLED = False
GITHUB_ENABLED = False
GITHUB_PROJECT_CONNECTION_ID = ""  # Required when GITHUB_ENABLED is True

In [ ]:
from agents._base.client import reset_client
from agents.registry import REGISTRY

# Reset singleton to ensure we use the current .env endpoint
reset_client()

# Always deploy/update the agent via the factory.
# This ensures the latest instructions + tools are pushed to Foundry.
entry = REGISTRY.get_agent(AGENT_NAME)
config = entry.config_class()

# Apply integration toggle overrides if the toggle cell was run;
# otherwise fall back to the agent's config.py defaults.
if "CODE_INTERPRETER_ENABLED" in dir():
    config.code_interpreter_enabled = CODE_INTERPRETER_ENABLED
if "WEB_SEARCH_ENABLED" in dir():
    config.web_search_enabled = WEB_SEARCH_ENABLED
if "GITHUB_ENABLED" in dir():
    config.github_enabled = GITHUB_ENABLED
if "GITHUB_PROJECT_CONNECTION_ID" in dir() and GITHUB_PROJECT_CONNECTION_ID:
    config.github_project_connection_id = GITHUB_PROJECT_CONNECTION_ID

# Verify the endpoint matches what the notebook is using
print(f"Factory endpoint: {config.azure_ai_project_endpoint}")
print(f"Code Interpreter: {config.code_interpreter_enabled}")
print(f"Web Search:       {config.web_search_enabled}")
print(f"GitHub MCP:       {config.github_enabled}")

agent = entry.factory(config)
print(f"\n✓ Agent ready: {agent.name} (version: {agent.version})")

# List all agents on this endpoint to confirm it's visible
agents_list = list(project_client.agents.list())
print(f"\nAll agents on this endpoint ({len(agents_list)}):")
for a in agents_list:
    marker = " ← THIS ONE" if a.name == agent.name else ""
    print(f"  - {a.name} (id: {a.id}){marker}")

In [ ]:
# Diagnostic: print the portal URL for this project so you can cross-reference
endpoint = config.azure_ai_project_endpoint.rstrip("/")
# Extract the hub host and project name from the endpoint
# Endpoint format: https://<hub>.services.ai.azure.com/api/projects/<project>
parts = endpoint.split("/api/projects/")
if len(parts) == 2:
    hub_url = parts[0]  # https://<hub>.services.ai.azure.com
    project_name = parts[1]
    print(f"Hub endpoint:  {hub_url}")
    print(f"Project name:  {project_name}")
    print(f"\n→ In Azure AI Foundry portal, make sure you're viewing project '{project_name}'")
    print(f"  then go to: Build → Agents")
    print(f"\n→ Agent name to look for: {agent.name}")
else:
    print(f"Endpoint: {endpoint}")
    print(f"Agent name: {agent.name}")

## Step 2: Create a Conversation and Send a Message

In [ ]:
import importlib, json

# Get OpenAI client for conversations/responses
openai_client = project_client.get_openai_client()

# Create a conversation with an initial message — change the content to test different prompts
USER_MESSAGE = "Hello! What can you help me with?"

conversation = openai_client.conversations.create(
    items=[{"type": "message", "role": "user", "content": USER_MESSAGE}],
)
print(f"✓ Conversation created: {conversation.id}")
print(f"✓ Message sent: {USER_MESSAGE}")

## Step 3: Get the Agent's Response

In [ ]:
# Load the agent's tool functions for function call handling
module_name = AGENT_NAME.replace("-", "_")
tool_functions = {}
try:
    tools_module = importlib.import_module(f"agents.{module_name}.tools")
    for tool in getattr(tools_module, "TOOLS", []):
        func = getattr(tools_module, tool.name, None)
        if func is None:
            for attr_name in dir(tools_module):
                sub = getattr(tools_module, attr_name)
                if hasattr(sub, tool.name):
                    func = getattr(sub, tool.name)
                    break
        if func and callable(func):
            tool_functions[tool.name] = func
    print(f"✓ Loaded {len(tool_functions)} tool function(s): {list(tool_functions.keys())}")
except ModuleNotFoundError:
    print("⚠ No tools module found for this agent")

# Send to the agent
agent_ref = {"agent_reference": {"name": agent.name, "type": "agent_reference"}}
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body=agent_ref,
)

# Handle function calls in a loop
MAX_ITERATIONS = 10
iteration = 0
while iteration < MAX_ITERATIONS:
    calls = [item for item in response.output if item.type == "function_call"]
    if not calls:
        break

    iteration += 1
    print(f"\n--- Function call round #{iteration} ---")
    results = []
    for call in calls:
        func = tool_functions.get(call.name)
        if func is None:
            print(f"  ✗ Unknown function: {call.name}")
            results.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps({"error": f"Unknown function: {call.name}"}),
            })
            continue

        arguments = json.loads(call.arguments) if call.arguments else {}
        print(f"  → {call.name}({arguments})")
        result = func(**arguments)
        output = json.dumps(result) if not isinstance(result, str) else result
        print(f"    ← {output[:200]}")
        results.append({
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": output,
        })

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body=agent_ref,
        input=results,
    )

print(f"\n{'='*60}")
print(f"Agent response:\n{response.output_text}")

In [ ]:
# Debug: dump ALL response output items to see what the agent actually did
print(f"Response ID: {response.id}")
print(f"Status: {response.status}")
print(f"\nOutput items ({len(response.output)}):")
for i, item in enumerate(response.output):
    print(f"\n--- Item {i} | type: {item.type} ---")
    if hasattr(item, "content"):
        for part in item.content:
            print(f"  {part.type}: {getattr(part, 'text', str(part))[:500]}")
    elif item.type == "function_call":
        print(f"  function: {item.name}({item.arguments[:200]})")
    else:
        print(f"  {item}")

## Step 4: Cleanup

Delete the conversation. The deployed agent is kept for future use.

In [ ]:
# Clean up the conversation (keeps the deployed agent)
openai_client.conversations.delete(conversation_id=conversation.id)
print("✓ Conversation deleted (agent kept for future use)")

## Next Steps

- Change the message content in Step 2 to test different prompts
- Toggle integrations: set `CODE_INTERPRETER_ENABLED`, `WEB_SEARCH_ENABLED`, or `GITHUB_ENABLED` to `True` and re-deploy
- Try a different agent: set `AGENT_NAME = "doc-assistant"` and re-run
- Add custom tools — see the [Custom Tools Guide](../docs/custom-tools-guide.md) for a full walkthrough
- See **03_scaffold_agent.ipynb** to create a brand new agent